# Notebook 03 — Joint Training
**Phase 2-3:** Freeze decoder, add LSTM head, train with joint loss.
L = MSE(reconstruction) + lambda * CrossEntropy(classification)

## 0. Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt
import torch

from src.config import (
    LAMBDA, JOINT_EPOCHS, MODELS_DIR, PLOTS_DIR, SEED, CLASSES
)
from src.model import Encoder, Decoder, LSTMHead, JointModel, freeze_decoder
from src.dataset import load_npy_split
from src.train import joint_train, plot_joint_loss, joint_train_lambda_sweep

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"PyTorch: {torch.__version__}  Device: {DEVICE}")

PyTorch: 2.13.0+cu130  Device: cuda


## 1. Load Data

In [2]:
X_train, y_train = load_npy_split("train")
X_val,   y_val   = load_npy_split("val")

print(f"Train: {X_train.shape}  labels: {np.bincount(y_train)}")
print(f"Val  : {X_val.shape}   labels: {np.bincount(y_val)}")

Train: (14658, 8, 64, 64, 3)  labels: [3000 2766 3000 2892 3000]
Val  : (5672, 8, 64, 64, 3)   labels: [3000  299 1030  145 1198]


## 2. Load Pre-trained Weights and Freeze Decoder

In [3]:
encoder = Encoder()
decoder = Decoder()
encoder.load_state_dict(torch.load(MODELS_DIR / "encoder_pretrained.pth", map_location=DEVICE))
decoder.load_state_dict(torch.load(MODELS_DIR / "decoder_pretrained.pth", map_location=DEVICE))

freeze_decoder(decoder)

trainable = sum(p.numel() for p in decoder.parameters() if p.requires_grad)
total     = sum(p.numel() for p in decoder.parameters())
print(f"Decoder trainable: {trainable}  frozen: {total - trainable}")

[INFO] Decoder frozen (8 parameter tensors).
Decoder trainable: 0  frozen: 2270819


## 3. Build Joint Model

In [4]:
lstm_head   = LSTMHead()
joint_model = JointModel(encoder, decoder, lstm_head)

# Encoder is frozen inside joint_train — only LSTM head (206K params) trains.
# This prevents the encoder from overfitting training-video appearances.
encoder_params = sum(p.numel() for p in joint_model.encoder.parameters())
lstm_params    = sum(p.numel() for p in joint_model.lstm_head.parameters())
frozen_total   = sum(p.numel() for p in joint_model.decoder.parameters())
print(f"Encoder params (frozen during joint train): {encoder_params:,}")
print(f"LSTM head params (trainable)              : {lstm_params:,}")
print(f"Decoder params (frozen)                   : {frozen_total:,}")

Encoder params (frozen during joint train): 2,190,656
LSTM head params (trainable)              : 206,213
Decoder params (frozen)                   : 2,270,819


## 4. Joint Training (Lambda = 0.5)

In [5]:
history = joint_train(joint_model, X_train, y_train, X_val, y_val,
                      lam=LAMBDA, device=DEVICE)

Epoch 01/30  loss=0.7070 recon=0.0288 cls=1.3564 acc=0.480 | val_loss=0.7687 val_acc=0.306
Epoch 02/30  loss=0.5547 recon=0.0288 cls=1.0518 acc=0.624 | val_loss=0.9080 val_acc=0.210
Epoch 03/30  loss=0.4685 recon=0.0288 cls=0.8795 acc=0.702 | val_loss=1.0779 val_acc=0.210
Epoch 04/30  loss=0.3998 recon=0.0288 cls=0.7419 acc=0.751 | val_loss=1.0741 val_acc=0.129
Epoch 05/30  loss=0.3552 recon=0.0288 cls=0.6528 acc=0.787 | val_loss=1.0953 val_acc=0.234
Epoch 06/30  loss=0.3130 recon=0.0288 cls=0.5683 acc=0.816 | val_loss=1.2861 val_acc=0.193
Epoch 07/30  loss=0.2826 recon=0.0288 cls=0.5077 acc=0.835 | val_loss=1.2661 val_acc=0.174
Epoch 08/30  loss=0.2505 recon=0.0288 cls=0.4432 acc=0.861 | val_loss=1.2450 val_acc=0.217
Epoch 09/30  loss=0.2331 recon=0.0288 cls=0.4085 acc=0.874 | val_loss=1.2848 val_acc=0.204
Epoch 10/30  loss=0.2204 recon=0.0288 cls=0.3831 acc=0.883 | val_loss=1.2368 val_acc=0.225
[INFO] Early stopping at epoch 11


## 5. Loss and Accuracy Curves

In [6]:
plot_joint_loss(history, save_path=PLOTS_DIR / "joint_loss.png")
plt.show()

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(history["train_acc"], label="Train Accuracy", linewidth=1.5)
ax.plot(history["val_acc"],   label="Val Accuracy",   linewidth=1.5, linestyle="--")
ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy")
ax.set_title("Classification Accuracy During Joint Training", fontweight="bold")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "joint_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()

[OK] Plot saved: /home/mjl/softwarica/ANN/outputs/plots/joint_loss.png


/tmp/ipykernel_36123/1791857050.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Lambda Sensitivity Analysis

In [ ]:
print("\nRunning lambda sensitivity analysis (trains 3 models)...")
lambda_results = joint_train_lambda_sweep(
    X_train, y_train, X_val, y_val,
    lambdas=(0.1, 0.5, 1.0), device=DEVICE
)

lambdas   = list(lambda_results.keys())
val_losses = [lambda_results[l]["val_loss"] for l in lambdas]
val_accs   = [lambda_results[l]["val_acc"]  for l in lambdas]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.5))
ax1.bar([str(l) for l in lambdas], val_losses, color="#4472C4")
ax1.set_title("Val Loss by Lambda", fontweight="bold")
ax1.set_xlabel("Lambda"); ax1.set_ylabel("Validation Loss")
ax2.bar([str(l) for l in lambdas], val_accs, color="#ED7D31")
ax2.set_title("Val Accuracy by Lambda", fontweight="bold")
ax2.set_xlabel("Lambda"); ax2.set_ylabel("Validation Accuracy")
plt.suptitle("Lambda Sensitivity Analysis", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "lambda_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()


Running lambda sensitivity analysis (trains 3 models)...

Lambda = 0.1
[INFO] Decoder frozen (8 parameter tensors).
Epoch 01/30  loss=0.1676 recon=0.0288 cls=1.3875 acc=0.459 | val_loss=0.1902 val_acc=0.327
Epoch 02/30  loss=0.1338 recon=0.0288 cls=1.0495 acc=0.625 | val_loss=0.2084 val_acc=0.251
Epoch 03/30  loss=0.1136 recon=0.0288 cls=0.8473 acc=0.710 | val_loss=0.2117 val_acc=0.230
Epoch 04/30  loss=0.1007 recon=0.0288 cls=0.7187 acc=0.758 | val_loss=0.2649 val_acc=0.118
Epoch 05/30  loss=0.0901 recon=0.0288 cls=0.6132 acc=0.800 | val_loss=0.2406 val_acc=0.171
Epoch 06/30  loss=0.0833 recon=0.0288 cls=0.5455 acc=0.824 | val_loss=0.2889 val_acc=0.127
Epoch 07/30  loss=0.0775 recon=0.0288 cls=0.4869 acc=0.844 | val_loss=0.3236 val_acc=0.125
Epoch 08/30  loss=0.0708 recon=0.0288 cls=0.4199 acc=0.868 | val_loss=0.3556 val_acc=0.117
Epoch 09/30  loss=0.0678 recon=0.0288 cls=0.3898 acc=0.877 | val_loss=0.3467 val_acc=0.095
Epoch 10/30  loss=0.0657 recon=0.0288 cls=0.3687 acc=0.884 | val

## 7. Summary

In [ ]:
joint_model.load_state_dict(
    torch.load(MODELS_DIR / "joint_model_best.pth", map_location=DEVICE))
print(f"Best val accuracy: {max(history['val_acc']):.4f}")
print(f"Best val loss    : {min(history['val_loss']):.4f}")
print("\n[DONE] Joint model saved: joint_model_best.pth")